# 04 — AutoAI Experiment + WML Deployment
**PIX Fraud Intelligence | IBM Portfolio Project**

**Goals:**
- Configure and run an AutoAI experiment via the `ibm_watsonx_ai` SDK
- Retrieve the pipeline leaderboard
- Deploy the best pipeline to a Watson Machine Learning endpoint
- Save deployment metadata for the evaluation notebook

> **Note:** AutoAI consumes CUH from your Watson Studio Lite quota (50 CUH/month).  
> A full experiment run uses ~5-10 CUH. Run sparingly.

> **WML Lite quota:** 20 CUH/month. The deployed endpoint persists without consuming  
> CUH — only predictions consume quota.

In [ ]:
import sys
sys.path.append('..')

import json
import pandas as pd
from ibm_watsonx_ai import APIClient, Credentials
import warnings
warnings.filterwarnings('ignore')

## 1. Connect to Watson Machine Learning

In [ ]:
with open('../config/credentials.json') as f:
    creds = json.load(f)

credentials = Credentials(
    api_key=creds['wml']['apikey'],
    url=creds['wml']['url']
)

client = APIClient(credentials, space_id=creds['wml']['space_id'])
print('WML client ready.')
print(f'ibm_watsonx_ai version: {client.version}')

## 2. Upload Training Data to IBM Cloud Object Storage

In [ ]:
# AutoAI reads training data from COS or inline
# We use the inline approach with the processed CSV
train_df = pd.read_csv('../data/processed/transactions_features.csv')
train_df = train_df[train_df['split'] == 'train'].drop(columns='split')
print(f'Training rows: {len(train_df):,} | Fraud rate: {train_df["Class"].mean():.4%}')

# Save training CSV for upload
train_df.to_csv('../data/processed/autoai_train.csv', index=False)

# Upload to WML data assets
asset_details = client.data_assets.create(
    name='pix_fraud_train',
    file_path='../data/processed/autoai_train.csv'
)
training_data_asset_uid = client.data_assets.get_id(asset_details)
print(f'\nData asset uploaded. UID: {training_data_asset_uid}')

## 3. Configure AutoAI Experiment
AutoAI automatically:
- Selects best algorithms (XGBoost, LightGBM, Random Forest, etc.)
- Applies hyperparameter optimization
- Generates up to 4 pipeline variants per algorithm
- Ranks them by the chosen metric (F1 — best for imbalanced data)

In [ ]:
from ibm_watsonx_ai.experiment import AutoAI
from ibm_watsonx_ai.helpers.connections import DataConnection

experiment = AutoAI(credentials, space_id=creds['wml']['space_id'])

pipeline_optimizer = experiment.optimizer(
    name='PIX Fraud Detection — AutoAI',
    prediction_type=AutoAI.PredictionType.BINARY,
    prediction_column='Class',
    positive_label=1,
    scoring=AutoAI.Metrics.F1_SCORE,
    include_only_estimators=[
        AutoAI.ClassificationAlgorithms.XGB,
        AutoAI.ClassificationAlgorithms.LGBM,
        AutoAI.ClassificationAlgorithms.RF,
    ],
    max_number_of_estimators=3,
    train_sample_rows_test_size=0.1,
)

print('AutoAI optimizer configured:')
print(f'  Task: Binary Classification')
print(f'  Target: Class (1 = fraud)')
print(f'  Metric: F1 Score')
print(f'  Algorithms: XGBoost, LightGBM, Random Forest')

## 4. Run AutoAI Experiment
> This cell takes 15-30 minutes. Monitor progress in Watson Studio UI.

In [ ]:
run_details = pipeline_optimizer.fit(
    training_data_reference=[
        DataConnection(data_asset_id=training_data_asset_uid)
    ],
    background_mode=False  # blocks until done
)
print('\nAutoAI experiment complete!')
print(f'Run ID: {run_details["metadata"]["id"]}')

## 5. Inspect Pipeline Leaderboard

In [ ]:
summary = pipeline_optimizer.summary()
print('Pipeline Leaderboard:')
print(summary[['Pipeline Name', 'Enhancements', 'Estimator', 'F1 score', 'Average precision score']]
      .sort_values('F1 score', ascending=False)
      .to_string())

## 6. Deploy Best Pipeline to WML Endpoint

In [ ]:
best_pipeline_name = summary.index[0]  # highest F1
print(f'Best pipeline: {best_pipeline_name}')

# Store best pipeline as WML model
best_pipeline = pipeline_optimizer.get_pipeline(best_pipeline_name)

model_details = pipeline_optimizer.get_pipeline_details(best_pipeline_name)
stored_model = client.repository.store_model(
    model=best_pipeline,
    meta_props={
        client.repository.ModelMetaNames.NAME: 'PIX Fraud Detector — Best AutoAI Pipeline',
        client.repository.ModelMetaNames.TYPE: 'scikit-learn_1.1',
        client.repository.ModelMetaNames.SOFTWARE_SPEC_UID:
            client.software_specifications.get_id_by_name('runtime-23.1-py3.10'),
    }
)
model_uid = client.repository.get_model_id(stored_model)
print(f'Model stored. UID: {model_uid}')

# Deploy
deployment = client.deployments.create(
    artifact_uid=model_uid,
    meta_props={
        client.deployments.ConfigurationMetaNames.NAME: 'PIX Fraud Detector — Online Endpoint',
        client.deployments.ConfigurationMetaNames.ONLINE: {},
    }
)
deployment_uid = client.deployments.get_id(deployment)
scoring_url = client.deployments.get_scoring_href(deployment)

print(f'\nDeployment UID: {deployment_uid}')
print(f'Scoring URL:    {scoring_url}')

## 7. Save Deployment Metadata

In [ ]:
# Persist for notebook 05
deployment_meta = {
    'model_uid': model_uid,
    'deployment_uid': deployment_uid,
    'scoring_url': scoring_url,
    'best_pipeline': best_pipeline_name,
}

with open('../config/deployment_meta.json', 'w') as f:
    json.dump(deployment_meta, f, indent=2)

print('Deployment metadata saved to config/deployment_meta.json')
print(json.dumps(deployment_meta, indent=2))

## 8. Quick Smoke Test
Score the first 3 test rows against the deployed endpoint.

In [ ]:
test_df = pd.read_csv('../data/processed/transactions_features.csv')
test_df = test_df[test_df['split'] == 'test'].drop(columns=['split', 'Class'])
sample = test_df.head(3)

payload = {'input_data': [{'values': sample.values.tolist()}]}
result = client.deployments.score(deployment_uid, payload)

print('Smoke test predictions:')
for i, pred in enumerate(result['predictions'][0]['values']):
    print(f'  Row {i}: prediction={pred[0]}, probability={pred[1]}')